# RL Post-Training with Hugging Face Transformer Reinforcement Learning (TRL) and Group Relative Policy Optimization (GRPO)

This notebook builds on TRL's [GRPO Trainer](https://huggingface.co/docs/trl/en/grpo_trainer) example to fine-tune the `Qwen/Qwen2.5-0.5B` model to answer math questions using the DeepMath 103k dataset. Ray Train scales this implementation efficiently across multiple GPUs without changing training logic. This notebook consists of the following steps:

1. [Set up Ray](#set-up)
2. [Quick Summary of GRPO and the DeepMaths dataset](#quick-summary)
3. [Running TRL with Ray Train](#training)

To install all the necessary dependencies, remove the `#` and run the following line. This notebook uses `transformers==4.57.6`:

In [ ]:
#! pip install "trl[vllm]" "math_verify" "transformers==4.57.6"

(set-up)=

## Set up Ray

Use `ray.init()` to initialize a local cluster. By default, this cluster contains only the machine you are running this notebook on. You can also run this notebook on an Anyscale cluster.

In [ ]:
import ray

ray.init()

Use `ray.cluster_resources()` to check which resources your cluster has access to. If you're running this notebook on your local machine or Google Colab, you should see the number of CPU cores and GPUs available to you.

In [3]:
from pprint import pprint

pprint(ray.cluster_resources())

{'CPU': 48.0,
 'GPU': 4.0,
 'accelerator_type:L4': 1.0,
 'anyscale/accelerator_shape:4xL4': 1.0,
 'anyscale/node-group:head': 1.0,
 'anyscale/provider:aws': 1.0,
 'anyscale/region:us-west-2': 1.0,
 'memory': 206158430208.0,
 'node:10.0.8.245': 1.0,
 'node:__internal_head__': 1.0,
 'object_store_memory': 58499506176.0}


This notebook performs RL post-training on a pre-trained Qwen2.5 model to answer math questions using the [DeepMaths dataset](https://arxiv.org/abs/2504.11456) and [GRPO](https://arxiv.org/abs/2402.03300). Ray Train parallelizes this training efficiently across multiple GPUs and/or nodes with fault-tolerance, observability, and more.

Change these two variables to control whether the training uses CPUs or GPUs, and how many workers to spawn. Each worker claims one CPU or GPU, so avoid requesting more resources than are available. By default, the training runs with one GPU worker.

In [4]:
use_gpu = True  # set this to `False` to run on CPUs
num_workers = 4  # set this to the number of GPUs or CPUs you want to use

(quick-summary)=

## Short Summary of the DeepMaths dataset and GRPO algorithm

The DeepMaths-103k dataset [He et al., 2025](https://arxiv.org/abs/2504.11456) consists of challenging questions across a wide range of mathematical subjects including Algebra, Calculus, Number Theory, Geometry, Probability, and Discrete Mathematics. These questions are more heavily distributed towards Levels 5 to 9 compared to alternative datasets, making them highly complex and difficult to solve without specialist training.

Each question includes a prompt for the model and a solution in LaTeX format, which provides a consistent way to represent complex mathematical equations.

In [5]:
from datasets import load_dataset

dataset = load_dataset("trl-lib/DeepMath-103K")

pprint(dataset["train"][0])
pprint(dataset["test"][0])

{'prompt': [{'content': 'Is it possible to construct an uncountable set of '
                        'subsets of the positive integers, \\( T \\), such '
                        'that for any two subsets \\( C, D \\) in \\( T \\), '
                        'either \\( C \\) is a subset of \\( D \\) or \\( D '
                        '\\) is a subset of \\( C \\)? Provide a justification '
                        'for your answer.',
             'role': 'user'}],
 'solution': 'Yes'}
{'prompt': [{'content': 'Evaluate the integral \\( \\int_S y \\, ds \\) where '
                        '\\( S \\) is the region in the plane \\( z = 1 + y '
                        '\\) that lies inside the cone \\( z = \\sqrt{2(x^2 + '
                        'y^2)} \\). Determine the bounds of integration and '
                        'compute the integral.',
             'role': 'user'}],
 'solution': '$2\\pi$'}


To RL fine-tune the model for math questions, the TRL GRPO implementation from [the original paper](https://arxiv.org/abs/2402.03300) handles the optimization.

For a deeper summary of the algorithm, the [TRL GRPO page](https://huggingface.co/docs/trl/en/grpo_trainer#looking-deeper-into-the-grpo-method) provides a more detailed description.

(training)=

## Using Hugging Face TRL with Ray Train

Compared to supervised pre-training, which trains models to minimize next-token prediction error, RL-based post-training maximizes reward from a prompt. Defining a reward function is therefore crucial to measure success.

The `trl.rewards.accuracy_reward` function checks whether the model's answer matches the answer from the dataset. These answers are in LaTeX format, requiring a parsing step to compare them. The default `trl.rewards.accuracy_reward` has timeouts on the `parse` and `verify` functions that are incompatible with Ray. The function below adds `parsing_timeout=0` and `timeout_seconds=0` to prevent this clash.

In [6]:
from latex2sympy2_extended import NormalizationConfig
from math_verify import LatexExtractionConfig, parse, verify

def accuracy_reward(completions, solution, **kwargs):
    """Reward function that checks mathematical accuracy.

    This is a copy of `trl.rewards.accuracy_reward`.
    The only difference is `parse(..., parsing_timeout=0)` and `verify(..., timeout_seconds=0)`
    to avoid `signal.alarm()` issues with ray.
    """
    contents = [completion[0]["content"] for completion in completions]
    rewards = []
    for content, sol in zip(contents, solution, strict=True):
        gold_parsed = parse(sol, parsing_timeout=0)
        if len(gold_parsed) != 0:
            # We require the answer to be provided in correct latex (no malformed operators)
            answer_parsed = parse(
                content,
                extraction_config=[
                    LatexExtractionConfig(
                        normalization_config=NormalizationConfig(units=True),
                        # Ensures that boxed is tried first
                        boxed_match_priority=0,
                        try_extract_without_anchor=False,
                    )
                ],
                extraction_mode="first_match",
                parsing_timeout=0,
            )
            reward = float(verify(gold_parsed, answer_parsed, timeout_seconds=0))
        else:
            # If the gold solution cannot be parsed, we assign `None` to skip this example
            reward = None
        rewards.append(reward)

    return rewards


The training function runs on all workers in parallel. It builds a `GRPOTrainer` that implements the GRPO algorithm.

The `vllm_mode="colocate"` setting gives each worker its own vLLM instance using 30% of GPU memory. The alternative `"server"` mode dedicates one worker to generation with vLLM while the remaining workers handle learning, but this introduces inter-GPU communication overhead that reduces throughput. Read [this blog post](https://huggingface.co/blog/vllm-colocate) for more information.

In [11]:
from ray.train.huggingface.transformers import RayTrainReportCallback, prepare_trainer
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset

def train_func(config):
    # Datasets
    dataset = load_dataset("trl-lib/DeepMath-103K", split="train")

    # Use vllm_mode="colocate" to allow easy scaling as each GPU handles their own vLLM instance
    training_args = GRPOConfig(
        per_device_train_batch_size=4, use_vllm=True, vllm_mode="colocate", vllm_gpu_memory_utilization=0.3,
    )

    # GRPO Trainer
    trainer = GRPOTrainer(
        model="Qwen/Qwen2.5-0.5B",
        args=training_args,
        reward_funcs=accuracy_reward,
        train_dataset=dataset,
    )

    # Report metrics and checkpoints to Ray Train
    trainer.add_callback(RayTrainReportCallback())

    # Prepare your trainer for Ray Data integration
    trainer = prepare_trainer(trainer)

    # Start Training
    trainer.train()

With your `train_func` complete, you can now instantiate the {class}`~ray.train.torch.TorchTrainer`. Aside from calling the function, set the `scaling_config`, which controls the amount of workers and resources used, and the `run_config` to configure checkpointing.

In [12]:
from ray.train.torch import TorchTrainer
from ray.train import RunConfig, ScalingConfig, CheckpointConfig

trainer = TorchTrainer(
    train_func,
    scaling_config=ScalingConfig(num_workers=num_workers, use_gpu=use_gpu),
    run_config=RunConfig(
        checkpoint_config=CheckpointConfig(
            num_to_keep=1,
            checkpoint_score_attribute="reward",
            checkpoint_score_order="max",
        ),
    ),
)

Finally, call the `fit` method to start training with Ray Train. Save the `Result` object to a variable so you can access metrics and checkpoints.

In [13]:
result = trainer.fit()

(TrainController pid=17806) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=17806) Attempting to start training worker group of size 4 with the following resources: [{'GPU': 1}] * 4
(RayTrainWorker pid=17905) Setting up process group for: env:// [rank=0, world_size=4]
(TrainController pid=17806) Started training worker group of size 4: 
(TrainController pid=17806) - (ip=10.0.8.245, pid=17905) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=17806) - (ip=10.0.8.245, pid=17909) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=17806) - (ip=10.0.8.245, pid=17910) world_rank=2, local_rank=2, node_rank=0
(TrainController pid=17806) - (ip=10.0.8.245, pid=17912) world_rank=3, local_rank=3, node_rank=0
(TrainController pid=17806) [State Transition] SCHEDULING -> RUNNING.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  7.04it/s]
Loading safetensors c

(RayTrainWorker pid=17905) {'loss': 0.0, 'grad_norm': 0.0, 'learning_rate': 9.999386941861653e-07, 'num_tokens': 48894.0, 'completions/mean_length': 225.4375, 'completions/min_length': 70.2, 'completions/max_length': 256.0, 'completions/clipped_ratio': 0.79375, 'completions/mean_terminated_length': 95.98833465576172, 'completions/min_terminated_length': 44.6, 'completions/max_terminated_length': 148.4, 'rewards/accuracy_reward/mean': 0.0, 'rewards/accuracy_reward/std': 0.0, 'reward': 0.0, 'reward_std': 0.0, 'frac_reward_zero_std': 1.0, 'sampling/sampling_logp_difference/mean': 0.019150843378156424, 'sampling/sampling_logp_difference/max': 0.3389609932899475, 'sampling/importance_sampling_ratio/min': 0.283931764960289, 'sampling/importance_sampling_ratio/mean': 0.9097023129463195, 'sampling/importance_sampling_ratio/max': 1.8704542517662048, 'entropy': 1.4526559472084046, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'cli

  0%|                                  | 20/146805 [00:58<100:33:45,  2.47s/it]


(RayTrainWorker pid=17905) {'loss': -0.0104, 'grad_norm': 7.047407150268555, 'learning_rate': 9.998705766152379e-07, 'num_tokens': 98995.0, 'completions/mean_length': 235.43125, 'completions/min_length': 94.6, 'completions/max_length': 256.0, 'completions/clipped_ratio': 0.8375, 'completions/mean_terminated_length': 116.325, 'completions/min_terminated_length': 69.0, 'completions/max_terminated_length': 159.1, 'rewards/accuracy_reward/mean': 0.00625, 'rewards/accuracy_reward/std': 0.025, 'reward': 0.00625, 'reward_std': 0.025, 'frac_reward_zero_std': 0.95, 'sampling/sampling_logp_difference/mean': 0.018179213255643846, 'sampling/sampling_logp_difference/max': 0.4839041829109192, 'sampling/importance_sampling_ratio/min': 0.2678584173321724, 'sampling/importance_sampling_ratio/mean': 0.9631744265556336, 'sampling/importance_sampling_ratio/max': 2.14500777721405, 'entropy': 1.3770145654678345, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/

  0%|                                  | 22/146805 [01:03<101:34:56,  2.49s/it]
(RayTrainWorker pid=17905) [RANK 0] All reward functions returned None for the following kwargs:
(RayTrainWorker pid=17905) {'solution': 'No', 'prompt': [{'content': 'Let $A$ be a finite-dimensional $k$-algebra that does not necessarily have a multiplicative identity. If the map \\( \\mu: A \\otimes A \\rightarrow A, \\ x \\otimes y \\mapsto xy \\) is surjective, does $A$ have a multiplicative identity?', 'role': 'user'}], 'completion': [{'role': 'assistant', 'content': 'The answer is yes.qualsystemmap\nOUND operation\nYou are a helpful assistant..getWriter\n.ExecuteReaderSystemReaderSystemReaderSystemReader\nFormatterSystemReaderSystemReaderSystemReader\nStreamReaderSystemReaderSystemReaderSystemReader\nStreamWriterSystemReaderSystemReaderSystemReader\nStreamWriterSystemReaderSystemReaderSystemReader\nStreamWriterSystemReaderSystemReaderSystemReader\nStreamWriterSystemReaderSystemReaderSystemReaderNamepass

...

  0%|                                  | 43/146805 [01:55<100:28:51,  2.46s/it]
(RayTrainWorker pid=17905) [RANK 0] All reward functions returned None for the following kwargs:
(RayTrainWorker pid=17905) {'solution': 'No', 'prompt': [{'content': 'Let $R$ be a ring and $I$ be a prime ideal of $R$. Suppose there is a surjective ring homomorphism $\\varphi: R \\to S$ such that $\\varphi(I)$ is a prime ideal of $S$. Does it follow that $I$ is generated by two elements? Answer yes or no.', 'role': 'user'}], 'completion': [{'role': 'assistant', 'content': "Do you need me to solve this question because I don't see what would be the perfect title for you?\nก่อนหน้าuser\nThank for your feedback. Yes, yes.PageRoute\n𬤊async\nPageRoute\n חוזר.radcom.com\nindrome.svg\nindrome.svg\nindrome.svg\nindrome.svg"}]}
(RayTrainWorker pid=17905) Please ensure that at least one reward function returns a valid reward.
2026-03-11 09:34:41,155	INFO data_parallel_trainer.py:289 -- Received SIGINT. Gracefully abor

SystemExit: 1

/home/ray/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


(TrainController pid=17806) [State Transition] RUNNING -> ABORTED.


(autoscaler +1h3m31s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.


You can use the returned `Result` object to access metrics and the Ray Train `Checkpoint` associated with the last iteration.

In [ ]:
result

## See also

* {doc}`Ray Train Examples <../../examples>` for more use cases
* {ref}`Ray Train User Guides <train-user-guides>` for how-to guides